# Crop Disease Detection Model Training

Training a MobileNetV2 model using the PlantVillage dataset.


In [1]:

!pip install kaggle

import os
os.environ['KAGGLE_USERNAME'] = "AbhiNavva"
os.environ['KAGGLE_KEY'] = "KGAT_83bdee6a9ce56756fc663e406e6c1222"

print('✅ Kaggle credentials configured.')

✅ Kaggle credentials configured.


In [2]:

!kaggle datasets download -d abdallahalidev/plantvillage-dataset
!unzip -q plantvillage-dataset.zip
print('✅ Dataset downloaded and extracted.')

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [01:57<00:00, 18.7MB/s]

✅ Dataset downloaded and extracted.


In [3]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import json
import os

from sklearn.metrics import classification_report, confusion_matrix

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

TensorFlow version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
data_dir = 'plantvillage dataset/color'
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset='training',
    seed=42,
    label_mode='int'
)

val_test_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset='validation',
    seed=42,
    label_mode='int'
)

val_test_size = tf.data.experimental.cardinality(val_test_ds).numpy()
val_size = val_test_size // 2
test_size = val_test_size - val_size

val_ds = val_test_ds.take(val_size)
test_ds = val_test_ds.skip(val_size)

print(f'Training batches:   {tf.data.experimental.cardinality(train_ds).numpy()}')
print(f'Validation batches: {tf.data.experimental.cardinality(val_ds).numpy()}')
print(f'Test batches:       {tf.data.experimental.cardinality(test_ds).numpy()}')
print(f'Number of classes:  {len(train_ds.class_names)}')
print(f'Class names:        {train_ds.class_names}')

Found 54305 files belonging to 38 classes.
Using 43444 files for training.
Found 54305 files belonging to 38 classes.
Using 10861 files for validation.
Training batches:   1358
Validation batches: 170
Test batches:       170
Number of classes:  38
Class names:        ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash_

In [5]:
class_names = train_ds.class_names
class_names_dict = {str(i): name for i, name in enumerate(class_names)}

with open('class_names.json', 'w') as f:
    json.dump(class_names_dict, f, indent=2)

print(f'✅ Saved {len(class_names)} classes to class_names.json')

✅ Saved 38 classes to class_names.json


In [6]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
])

print('✅ Data augmentation pipeline created.')

✅ Data augmentation pipeline created.


In [7]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(data_augmentation(x, training=True)), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

print('✅ Datasets normalized and prefetched.')

✅ Datasets normalized and prefetched.


## Feature Extraction


In [8]:
NUM_CLASSES = len(class_names)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze all layers

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
])

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 38)             │         9,766 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,595,686 (9.90 MB)

 Trainable params: 337,702 (1.29 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [14]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

print('🚀 Phase 1: Training with frozen backbone...')
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stopping]
)
print('✅ Phase 1 complete.')

🚀 Phase 1: Training with frozen backbone...
Epoch 1/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 542s 385ms/step - accuracy: 0.8192 - loss: 0.6016 - val_accuracy: 0.9004 - val_loss: 0.2975
Epoch 2/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 501s 369ms/step - accuracy: 0.8934 - loss: 0.3290 - val_accuracy: 0.9193 - val_loss: 0.2361
Epoch 3/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 501s 368ms/step - accuracy: 0.9044 - loss: 0.2865 - val_accuracy: 0.9281 - val_loss: 0.2152
Epoch 4/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 496s 365ms/step - accuracy: 0.9100 - loss: 0.2690 - val_accuracy: 0.9248 - val_loss: 0.2205
Epoch 5/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 507s 374ms/step - accuracy: 0.9140 - loss: 0.2561 - val_accuracy: 0.9353 - val_loss: 0.1990
Epoch 6/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 512s 377ms/step - accuracy: 0.9162 - loss: 0.2459 - val_accuracy: 0.9250 - val_loss: 0.2124
Epoch 7/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 521s 383ms/step - accuracy: 0.9226 - loss: 0.2329 - val_accuracy: 0.9307 - val_loss: 0.1953
Epoch 8/10
1358/13

In [ ]:
model.save('crop_disease_model_phase1.h5')
print('✅ Phase 1 model saved successfully!')

✅ Phase 1 model saved successfully! You can now safely restart your PC.


In [10]:
import tensorflow as tf
model = tf.keras.models.load_model('crop_disease_model_phase1.h5')
base_model = model.layers[0] # Grab the MobileNetV2 base model from the loaded model
print('✅ Phase 1 model loaded successfully! Ready for Phase 2.')

✅ Phase 1 model loaded successfully! Ready for Phase 2.


## Fine-Tuning


In [11]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping_ft = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

print('🚀 Phase 2: Fine-tuning last 30 layers...')
history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stopping_ft]
)
print('✅ Phase 2 complete.')

🚀 Phase 2: Fine-tuning last 30 layers...
Epoch 1/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 571s 401ms/step - accuracy: 0.8550 - loss: 0.5181 - val_accuracy: 0.9274 - val_loss: 0.2553
Epoch 2/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 587s 389ms/step - accuracy: 0.9486 - loss: 0.1619 - val_accuracy: 0.9469 - val_loss: 0.1677
Epoch 3/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 528s 389ms/step - accuracy: 0.9635 - loss: 0.1140 - val_accuracy: 0.9517 - val_loss: 0.1695
Epoch 4/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 522s 384ms/step - accuracy: 0.9707 - loss: 0.0921 - val_accuracy: 0.9583 - val_loss: 0.1414
Epoch 5/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 511s 376ms/step - accuracy: 0.9730 - loss: 0.0809 - val_accuracy: 0.9614 - val_loss: 0.1269
Epoch 6/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 521s 383ms/step - accuracy: 0.9777 - loss: 0.0687 - val_accuracy: 0.9658 - val_loss: 0.1126
Epoch 7/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 559s 381ms/step - accuracy: 0.9795 - loss: 0.0621 - val_accuracy: 0.9647 - val_loss: 0.1187
Epoch 8/10
1358/1358 

In [12]:
test_loss, test_accuracy = model.evaluate(test_ds)
print(f'\n📊 Test Loss:     {test_loss:.4f}')
print(f'📊 Test Accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)')

170/170 ━━━━━━━━━━━━━━━━━━━━ 22s 107ms/step - accuracy: 0.9701 - loss: 0.0977

📊 Test Loss:     0.0977
📊 Test Accuracy: 0.9701 (97.01%)


In [ ]:
model.save('crop_disease_model.h5')
print('✅ Model saved to crop_disease_model.h5')
print('Model and labels saved.')

✅ Model saved to crop_disease_model.h5

🎉 All done! The files are saved locally:
   - crop_disease_model.h5  (trained model)
   - class_names.json       (class label mapping)


In [14]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('crop_disease_model.h5', '/content/drive/MyDrive/')
shutil.copy('class_names.json', '/content/drive/MyDrive/')
print("✅ Files successfully copied to your Google Drive!")

Mounted at /content/drive
✅ Files successfully copied to your Google Drive!
